## Validação do Bronze

Este notebook confere se os arquivos brutos da SUSEP foram carregados no
Volume sem perda de dados.

A comparação é feita contra o `manifest_bronze.json` — um arquivo de
controle gerado localmente a partir dos arquivos originais, antes do
upload (script de geração não versionado, uso único). Ele registra a
contagem de linhas, delimitador e encoding de cada arquivo na fonte,
servindo como referência independente: se a contagem lida aqui no
Databricks bater com a do manifesto, a ingestão foi bem-sucedida.

In [0]:
import json

caminho_manifesto = "/Workspace/Users/joelgjunior@gmail.com/mvp_engenharia_de_dados/manifest_bronze.json"
caminho_base = "/Volumes/susep_capitalizacao/bronze/raw_susep_capitalizacao"

with open(caminho_manifesto) as f:
    manifesto = json.load(f)

resultados = []

for nome_arquivo, info in manifesto["arquivos"].items():
    df = spark.read.csv(
        f"{caminho_base}/{nome_arquivo}",
        header=True,
        sep=info["delimitador"],
        encoding=info["encoding"],
    )
    contagem = df.count()
    esperado = info["linhas"]
    status = "OK" if contagem == esperado else "DIVERGENTE"
    resultados.append({"arquivo": nome_arquivo, "df": df, "contagem": contagem, "esperado": esperado, "status": status})

# --- Resumo primeiro ---
print("=== RESUMO ===")
for r in resultados:
    print(f"{r['status']:<11} {r['arquivo']}")

# --- Detalhe depois ---
print("\n=== DETALHE ===")
for r in resultados:
    print(f"\n{r['arquivo']}: {r['contagem']:,} linhas (esperado {r['esperado']:,}) -> {r['status']}")
    r["df"].printSchema()